In [13]:
import itertools
import json
import os


In [14]:
data_dir = "./data"
raw_dir = os.path.join(data_dir, "raw")


In [15]:
def load_ndjson(path):
	lines = []

	with open(path, "r", encoding="utf-8") as f:
		for i, line in enumerate(f):
			line = line.strip()
			if line:
				try:
					lines.append(json.loads(line))
				except json.JSONDecodeError:
					print(f"Invalid JSON in {path} (line {i}): {line[:200]}")
	return lines

def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def load_traces(base_dir: str):
	gameplay_dir = os.path.join(base_dir, "Gameplay")

	traces = []
	for file in os.listdir(gameplay_dir):
		if file.endswith(".ndjson"):
			file_path = os.path.join(gameplay_dir, file)
			content = load_ndjson(file_path)
			traces.append(content)

	# Aplanar la lista de listas devuelta por load_ndjson
	return list(itertools.chain.from_iterable(traces))


In [16]:
GAME_START_OBJ_ID = "https://w3id.org/xapi/seriousgames/activity-types/serious-game/GameStart"


In [17]:
def get_users(traces: list):
	users = set()

	for trace in traces:
		user = trace.get("actor", {}).get("account", {}).get("name")

		if user:
			obj_id = trace.get("object", {}).get("id", "")
			if obj_id == GAME_START_OBJ_ID:
				users.add(user)

	return users

In [18]:
def get_users_test(data: dict) -> set:
    users = set()

    for user in data:
        if data[user] is not None:
            users.add(user)

    return users


In [19]:
def get_users_from_code_full(folder_path: str):
	users = set()

	for file_entry in os.scandir(folder_path):
		file_name = file_entry.name

		if "code" in file_name and file_name.endswith(".json"):
			code_data = load_json(file_entry.path)
			code_users = get_users_test(code_data)
			users = users | code_users

		elif "full" in file_name and file_name.endswith(".json"):
			full_data = load_json(file_entry.path)
			full_users = get_users_test(full_data)
			users = users | full_users

	return users


In [20]:
total_users = set()

for session_entry in os.scandir(raw_dir):
	session_dir = session_entry.path

	for folder_name in ("Gameplay", "Post", "Pre"):
		folder_path = os.path.join(session_dir, folder_name)

		if folder_name == "Gameplay":
			traces = load_traces(session_dir)
			users = get_users(traces)

		if folder_name == "Post":
			users = get_users_from_code_full(folder_path)

		elif folder_name == "Pre":
			users = get_users_from_code_full(folder_path)
			
		total_users.update(users)


In [21]:
print(total_users)


{'682b4a41c76d2e0023ed4b24_rvns', '682b4a41c76d2e0023ed4b24_iwjt', 'wxlh', '684837b0e48b5a00221a37d0_wasa', '682b4a72c76d2e0023ed4d05_bnrb', '684837b0e48b5a00221a37d0_xkmr', '682b4a72c76d2e0023ed4d05_kqud', '684837b0e48b5a00221a37d0_cpzn', '682b4a72c76d2e0023ed4d05_azng', '684837b0e48b5a00221a37d0_xuse', '682b4a41c76d2e0023ed4b24_eggn', '684837aae48b5a00221a37c9_xnjp', '684837aae48b5a00221a37c9_fwme', 'doxh', 'labv', '684837aae48b5a00221a37c9_grwp', '682b4a72c76d2e0023ed4d05_nsnn', '682b4a41c76d2e0023ed4b24_sxif', 'fivy', '684837b0e48b5a00221a37d0_mmng', '684837aae48b5a00221a37c9_dlez', 'mnkw', '684837b0e48b5a00221a37d0_lntj', '684837aae48b5a00221a37c9_ndkj', '684837aae48b5a00221a37c9_cgph', 'lbmd', '684837b0e48b5a00221a37d0_ossy', '682b4a41c76d2e0023ed4b24_vdon', 'mlsu', 'xpbo', 'eqzv', '682b4a72c76d2e0023ed4d05_ohow', '682b4a72c76d2e0023ed4d05_cvlf', '682b4a41c76d2e0023ed4b24_notf', '684837b0e48b5a00221a37d0_lnbt', '682b4a72c76d2e0023ed4d05_agog', '684837b0e48b5a00221a37d0_ryrm', '68

In [23]:
print(len(total_users))


131


In [ ]:
excluded_users = {
	"memogames",
	"682b4a72c76d2e0023ed4d05_igbm",
	"682b4a41c76d2e0023ed4b24_dmyj",
	"682b4a41c76d2e0023ed4b24_sxif",
	"682b4a72c76d2e0023ed4d05_agog"
}

valid_users = total_users - excluded_users


131


In [26]:
print(len(valid_users))


126
